# Flight Fare Prediction Using Machine Learning


| Step | Section |
|------|---------|
| 0 | Environment Setup |
| 1 | Problem Definition |
| 2 | Data Cleaning (10-step pipeline) |
| 3 | Feature Engineering & Train/Test Split |
| 4 | Exploratory Data Analysis |
| 5 | Baseline — Linear Regression |
| 6 | Advanced Modelling & Hyperparameter Tuning |
| 7 | Evaluation — Model Comparison & CV Report |
| 8 | Best Model Diagnostics |
| 9 | Feature Importance & SHAP |
| 10 | Save Artefacts |
| 11 | Business Insights |

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold

_cwd = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_cwd, _cwd.parent, _cwd.parent.parent]
     if (p / "src" / "__init__.py").exists()),
    _cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

from src.config import (
    DATA_PATH, OUTPUTS_DIR, RANDOM_STATE,
    CV_FOLDS, GRID_CV_FOLDS, N_ITER,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET,
)
from src.logger import setup_logging
from src.data_loader import load_data, inspect_data
from src.data_cleaner import (
    clean_pipeline, audit_missing_values,
    validate_value_ranges, validate_categorical_values,
)
from src.feature_engineering import engineer_features, build_preprocessor, split_and_transform
from src.eda import run_eda
from src.models import train_all_models, build_cv_report
from src.evaluation import (
    build_comparison_table, plot_cv_fold_scores,
    plot_best_model_dashboard, plot_predicted_vs_actual,
    plot_residuals, plot_learning_curve, plot_ridge_vs_lasso,
)

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
setup_logging(log_dir=OUTPUTS_DIR)
logger = logging.getLogger("notebook")
%matplotlib inline

---
## Step 1 — Problem Definition & Data Understanding

**Business Question:** Given a flight's route, airline, travel date, cabin class, and booking lead time — what will the total ticket fare (BDT) be?

**ML Task:** Supervised Regression | **Target:** `Total_Fare` (BDT)

**Assumption:** Seasonal labels (`Season`) are pre-assigned in the dataset and not derived independently from the date.

In [ ]:
df_raw = load_data(DATA_PATH)

In [ ]:
inspect_data(df_raw)

In [ ]:
df_raw.head(3)

---
## Step 2 — Data Cleaning & Validation

`clean_pipeline()` runs ten sequential steps — each assumes the previous has completed.

| Step | Function | Action |
|------|----------|--------|
| 1 | `standardise_column_names` | Rename raw CSV columns to snake_case |
| 2 | `validate_schema` | Fail fast if a required column is missing |
| 3 | `validate_dtypes` | Coerce `Date` → datetime; save `validation_dtypes.csv` |
| 4 | `validate_value_ranges` | Remove rows outside valid bounds; save `validation_ranges.csv` |
| 5 | `validate_categorical_values` | Audit unexpected values; save `validation_categoricals.csv` |
| 6 | `normalise_categorical_values` | Fix label inconsistencies — e.g. `'First Class'` → `'First'` |
| 7 | `remove_duplicates` | Drop exact duplicate rows |
| 8 | `audit_missing_values` | Pre-imputation null snapshot; save `validation_missing.csv` |
| 9 | `handle_missing_values` | Median (numerics) · mode (categoricals) · `'Unknown'` (Aircraft_Type) |
| 10 | `fix_invalid_fares` | Recompute `Total_Fare = Base_Fare + Tax_Surcharge`; drop fares < BDT 500 |

In [ ]:
df_clean = clean_pipeline(df_raw, OUTPUTS_DIR)

In [ ]:
post_audit = audit_missing_values(df_clean)

assert df_clean.isnull().sum().sum() == 0, "Nulls remain after cleaning — check handle_missing_values()"

post_audit

In [ ]:
_, range_report = validate_value_ranges(df_clean.copy())
range_report

In [ ]:
anomaly_df = validate_categorical_values(df_clean)
if not anomaly_df.empty:
    display(anomaly_df)

---
## Step 3 — Feature Engineering & Train/Test Split

| Feature | Source | Rationale |
|---------|--------|-----------|
| `Month` | `Date.month` | Seasonal pricing cycle |
| `Weekday` | `Date.dayofweek` | Mid-week flights are typically cheaper |
| `Stop_Type` | `Stop_Raw` | Binary Non-Stop vs With_Stop |

Preprocessor fitted on **X_train only** (prevents leakage): `StandardScaler` on numerics, `OneHotEncoder(handle_unknown='ignore')` on categoricals → 73 features. 80/20 split, `random_state=42`.

In [ ]:
df_feat = engineer_features(df_clean)

In [ ]:
preprocessor = build_preprocessor()

(
    X_train, X_test, y_train, y_test,
    X_train_proc, X_test_proc,
    feature_names, preprocessor,
    log_transformed
) = split_and_transform(df_feat, preprocessor)

---
## Step 4 — Exploratory Data Analysis

| Dashboard | Contents |
|-----------|---------|
| Fare Overview | Distribution · log-transformed · monthly trend · fare by season |
| Airline Analysis | Average fare by airline · spread boxplot |
| Route Intelligence | Top-10 popular routes · top-5 most expensive |
| Correlations | Pearson heatmap for numerical features |

In [ ]:
eda_figs = run_eda(df_feat, OUTPUTS_DIR)

In [ ]:
display(eda_figs['fare_overview'])

**Observation:**
- Total fare is heavily right-skewed — most tickets fall below BDT 100,000 with a long upper tail
- After `log1p` transformation, the distribution becomes near-symmetric (skew: 1.554 → -0.172)
- Monthly trend shows peaks in April/May (Eid-ul-Fitr) and July/August (Eid-ul-Adha); February is cheapest

The log transformation stabilises variance and satisfies linear model assumptions — justifying its use as the target encoding throughout this pipeline.

In [ ]:
display(eda_figs['airline_analysis'])

**Observation:**
- Premium international carriers (e.g. Turkish Airlines, Cathay Pacific) show the highest average fares
- US-Bangla has the tightest IQR — consistent value-carrier pricing with minimal surge behaviour
- Biman Bangladesh charges ~18% above the dataset average on domestic routes

Airline identity is a strong predictive feature: carrier choice alone explains a significant portion of fare variance, independent of route or season.

In [ ]:
season_stats = (
    df_feat.groupby('Season')['Total_Fare']
    .agg(median='median', mean='mean', std='std', count='count')
    .sort_values('median', ascending=False)
    .round(0)
)
# Calculate premium relative to the lowest-median season
min_median = season_stats['median'].min()
season_stats['premium_%'] = ((season_stats['median'] - min_median) / min_median * 100).round(1)
season_stats

**Observation:**
- Eid median fare is **+41.9%** above Regular (off-peak) season
- Hajj shows the highest spread and median — driven by concentrated demand on a narrow time window
- Winter Holidays sit between Eid/Hajj and Regular (+18.8%)

Seasonality is the single most actionable signal for dynamic pricing — advance-booking campaigns timed 6+ weeks before Eid and Hajj periods can capture the largest fare premium.

In [ ]:
display(eda_figs['correlations'])

**Observation:**
- `Base_Fare` and `Tax_Surcharge` correlate ~1.00 with `Total_Fare` — they are components of the target, not independent features
- `Duration_hrs` shows moderate positive correlation (r ≈ 0.65) — longer flights cost more
- `Days_Before_Departure` has a weak negative correlation — earlier bookings are slightly cheaper
- `Month` and `Weekday` show near-zero individual correlation, but contribute through interaction with seasonality

`Base_Fare` and `Tax_Surcharge` are excluded from the feature matrix to prevent target leakage.

In [ ]:
display(eda_figs['route_analysis'])

**Observation:**
- DAC→CGP (Dhaka → Chittagong) is the highest-volume domestic corridor
- International long-haul routes (e.g. DAC→LHR, DAC→JFK) dominate the most-expensive list
- DAC→CGP also shows the highest intra-airline price variance — a key opportunity for competitor monitoring

Route distance is a proxy for fare but not the primary driver — cabin class and airline explain more variance on the same route.

---
## Step 5 — Baseline Model: Linear Regression

No hyperparameters — trained in a single pass as a reference point before advanced modelling.

In [ ]:
lr_baseline = LinearRegression().fit(X_train_proc, y_train)
y_pred_lr   = lr_baseline.predict(X_test_proc)

r2_lr = r2_score(y_test, y_pred_lr)

if log_transformed:
    y_true_bdt_lr = np.expm1(np.array(y_test))
    y_pred_bdt_lr = np.expm1(np.array(y_pred_lr))
else:
    y_true_bdt_lr = np.array(y_test)
    y_pred_bdt_lr = np.array(y_pred_lr)

mae_lr  = mean_absolute_error(y_true_bdt_lr, y_pred_bdt_lr)
rmse_lr = mean_squared_error(y_true_bdt_lr, y_pred_bdt_lr) ** 0.5

In [ ]:
fig_lr_pred = plot_predicted_vs_actual(y_test, y_pred_lr, 'Linear Regression', OUTPUTS_DIR)
display(fig_lr_pred)

fig_lr_res = plot_residuals(y_test, y_pred_lr, 'Linear Regression', OUTPUTS_DIR)
display(fig_lr_res)

> **Observation:** The fan-shaped pattern in the residuals vs. predicted plot reveals
> **heteroscedasticity** — the model's errors grow proportionally with predicted fare.
> This is the hallmark of a nonlinear signal (seasonal surges, airline pricing tiers)
> that a global linear model cannot capture. It directly motivates regularised and tree-based models in Step 6.

---
## Step 6 — Advanced Modelling & Hyperparameter Tuning

| Model | Search Method | Fits | Reason |
|-------|--------------|------|--------|
| Linear Regression | None | — | Baseline |
| Ridge | GridSearchCV | 75 | Single α param — exhaustive is trivial |
| Lasso | GridSearchCV | 45 | Denser grid in L1 transition zone |
| Decision Tree | **GridSearchCV** | **300** | 5×4×3 grid — exhaustive feasible, misses nothing |
| Random Forest | RandomizedSearchCV (n_iter=50) | 250 | ~1,200 combos — full grid infeasible |
| XGBoost | RandomizedSearchCV (n_iter=50) | 250 | ~800k combos — only random sampling is practical |
| LightGBM | RandomizedSearchCV (n_iter=50) | 250 | Same scale as XGBoost |

All models share the same `KFold(n_splits=5, shuffle=True, random_state=42)` — identical fold splits eliminate split-randomness as a confounding variable.

### Search Spaces

In [ ]:
# ── Exact hyperparameter search spaces (read live from models.py) ─────────────
from src.models import get_grid_params, get_random_params
from src.config import N_ITER, GRID_CV_FOLDS
from math import prod

_grid   = get_grid_params()
_random = get_random_params()

print(f"{'='*65}")
print(f"SEARCH SPACES  (GridSearchCV folds={GRID_CV_FOLDS} | n_iter={N_ITER})")
print(f"{'='*65}\n")

print("GridSearchCV — EXHAUSTIVE\n")
for name, params in _grid.items():
    n_combos = prod(len(v) for v in params.values())
    print(f"  {name}  →  {n_combos} combos × {GRID_CV_FOLDS} folds = {n_combos * GRID_CV_FOLDS} fits")
    for p, v in params.items():
        print(f"    {p:30s}: {v}")
    print()

print(f"RandomizedSearchCV — n_iter={N_ITER}\n")
for name, params in _random.items():
    n_full = prod(len(v) for v in params.values())
    print(f"  {name}  →  {N_ITER} samples / ~{n_full:,} total combos × {GRID_CV_FOLDS} folds = {N_ITER * GRID_CV_FOLDS} fits")
    for p, v in params.items():
        print(f"    {p:30s}: {v}")
    print()

In [ ]:
kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

best_models, results = train_all_models(
    X_train_proc, y_train, X_test_proc, y_test, kf,
    log_transformed=log_transformed,
)

---
## Step 7 — Model Evaluation & Comparison

Models ranked by **CV R² Mean** (primary) → **CV R² Std** (lower = more stable) → **RMSE** (lower = better).  
CV R² is preferred over a single test-split score because 5-fold evaluation covers every row as both train and validation data, giving a lower-variance estimate of generalisation.

Per-fold CV columns: **Mean** (ranking criterion) · **Std** (stability, ✓ if < 0.03) · **Range** (Max−Min, flag if > 0.05) · **CI_lower/upper** (95% confidence interval)

In [ ]:
# ── Best hyperparameters — read directly from trained estimators ──────────────
_search_method = {
    "Linear Regression": "None (baseline)",
    "Ridge":             "GridSearchCV",
    "Lasso":             "GridSearchCV",
    "Decision Tree":     "GridSearchCV",
    "Random Forest":     "RandomizedSearchCV",
    "XGBoost":           "RandomizedSearchCV",
    "LightGBM":          "RandomizedSearchCV",
}
_tuned_keys = {
    "Linear Regression": [],
    "Ridge":             ["alpha"],
    "Lasso":             ["alpha", "max_iter"],
    "Decision Tree":     ["max_depth", "min_samples_leaf", "min_samples_split"],
    "Random Forest":     ["n_estimators", "max_depth", "max_features",
                          "min_samples_leaf", "min_samples_split"],
    "XGBoost":           ["n_estimators", "learning_rate", "max_depth",
                          "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"],
    "LightGBM":          ["n_estimators", "learning_rate", "num_leaves",
                          "max_depth", "subsample", "colsample_bytree"],
}

rows = []
for name, model in best_models.items():
    params = model.get_params()
    keys   = _tuned_keys.get(name, [])
    best_str = ", ".join(f"{k}={params[k]}" for k in keys if k in params) or "—"
    rows.append({"Model": name, "Search Method": _search_method[name], "Best Parameters": best_str})

display(pd.DataFrame(rows).set_index("Model"))

In [ ]:
# Build summary table sorted by CV R²
comparison = build_comparison_table(results)

# Account for log-target column names
r2_col = 'R² (BDT)' if 'R² (BDT)' in comparison.columns else 'R²'

comparison.style \
    .background_gradient(subset=[r2_col, 'CV R² Mean'], cmap='Greens') \
    .background_gradient(subset=['MAE (BDT)', 'RMSE (BDT)'], cmap='Reds_r') \
    .highlight_min(subset=['CV R² Std'], color='#d4edda') \
    .format({
        r2_col: '{:.4f}',
        'MAE (BDT)': '{:,.0f}',
        'RMSE (BDT)': '{:,.0f}',
        'CV R² Mean': '{:.4f}',
        'CV R² Std': '{:.4f}',
    })

In [ ]:
cv_report = build_cv_report(best_models, X_train_proc, y_train, kf)

In [ ]:
cv_df = plot_cv_fold_scores(best_models, X_train_proc, y_train, kf, OUTPUTS_DIR)
plt.show()

In [ ]:
# ── Styled per-fold CV report ─────────────────────────────────────────────────
cv_display_cols = (
    [f"Fold {i}" for i in range(1, CV_FOLDS + 1)]
    + ["Mean", "Std", "Min", "Max", "Range", "CI_lower", "CI_upper", "Stable"]
)
cv_display_cols = [c for c in cv_display_cols if c in cv_report.columns]

cv_report[cv_display_cols].sort_values("Mean", ascending=False).style \
    .background_gradient(subset=["Mean"], cmap="Greens") \
    .background_gradient(subset=["Std", "Range"], cmap="Reds_r") \
    .highlight_max(subset=["Mean"],  color="#c3e6cb") \
    .highlight_min(subset=["Std", "Range"], color="#d4edda") \
    .format({
        "Mean": "{:.4f}", "Std": "{:.4f}",
        "Min":  "{:.4f}", "Max": "{:.4f}", "Range": "{:.4f}",
        "CI_lower": "{:.4f}", "CI_upper": "{:.4f}",
        **{f"Fold {i}": "{:.4f}" for i in range(1, CV_FOLDS + 1)},
    })

In [ ]:
fig_reg = plot_ridge_vs_lasso(
    best_models["Ridge"], best_models["Lasso"],
    feature_names, OUTPUTS_DIR
)
display(fig_reg)

> **Regularisation insight:** Lasso zeroed **44 out of 73 features** — mainly minor booking-channel dummies and less-popular route one-hot columns. Ridge retains all 73 features but shrinks them proportionally, which is better suited to the multicollinear overlap between Airline and Route signals. Both achieve similar CV R², confirming the zeroed features carried minimal predictive power.

---
## Step 8 — Best Model Diagnostics

In [ ]:
best_name   = comparison.index[0]
best_model  = best_models[best_name]
y_pred_best = best_model.predict(X_test_proc)

if log_transformed:
    y_true_bdt = np.expm1(np.array(y_test))
    y_pred_bdt = np.expm1(np.array(y_pred_best))
else:
    y_true_bdt = np.array(y_test)
    y_pred_bdt = np.array(y_pred_best)

r2_best   = r2_score(y_test, y_pred_best)
r2_bdt    = r2_score(y_true_bdt, y_pred_bdt)
mae_best  = mean_absolute_error(y_true_bdt, y_pred_bdt)
rmse_best = mean_squared_error(y_true_bdt, y_pred_bdt) ** 0.5

In [ ]:
fig_diag = plot_best_model_dashboard(
    y_test, y_pred_best, best_model, best_name, feature_names,
    OUTPUTS_DIR, log_transformed=log_transformed,
)
display(fig_diag)

In [ ]:
fig_lc = plot_learning_curve(best_model, X_train_proc, y_train, kf, best_name, OUTPUTS_DIR)
display(fig_lc)

---
## Step 9 — Model Interpretation & Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    raw_imp = pd.Series(best_model.feature_importances_, index=feature_names)
elif hasattr(best_model, 'coef_'):
    raw_imp = pd.Series(np.abs(best_model.coef_), index=feature_names)
else:
    raw_imp = None

if raw_imp is not None:
    top10 = raw_imp.sort_values(ascending=False).head(10)
    top10_df = top10.reset_index()
    top10_df.columns = ['Feature', 'Importance_Score']
    top10_df['Importance_%'] = (top10_df['Importance_Score'] / raw_imp.sum() * 100).round(2)
    top10_df['Rank'] = range(1, 11)
    display(top10_df.set_index('Rank'))

### Optional Stretch — SHAP Values (Explainable AI)

In [ ]:
try:
    import shap

    explainer   = shap.TreeExplainer(best_model)
    shap_sample = X_test_proc[:500]
    shap_values = explainer.shap_values(shap_sample)

    shap.summary_plot(
        shap_values, shap_sample, feature_names=feature_names,
        plot_type='bar', max_display=15, show=False
    )
    plt.title(f'SHAP Global Feature Importance — {best_name}', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'shap_importance.png', dpi=150)
    plt.show()

    shap.summary_plot(
        shap_values, shap_sample, feature_names=feature_names,
        max_display=12, show=False
    )
    plt.title('SHAP Beeswarm — Per-Prediction Feature Impact', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'shap_beeswarm.png', dpi=150)
    plt.show()

except ImportError:
    logger.warning('shap not installed — run: pip install shap==0.45.1')

---
## Step 10 — Save Artefacts

In [ ]:
joblib.dump(best_model,   OUTPUTS_DIR / "best_model.pkl")
joblib.dump(preprocessor, OUTPUTS_DIR / "preprocessor.pkl")

comparison.to_csv(OUTPUTS_DIR / "model_metrics.csv")
cv_report.to_csv(OUTPUTS_DIR / "cv_report_full.csv")

---
## Step 11 — Business Insights & Stakeholder Recommendations

In [ ]:
insights = {
    'Finding': [
        'Eid seasons drive a 35–45% fare premium over off-peak',
        'Airline brand explains more fare variance than route distance',
        'Non-stop premium is modest (~10%) — stops are not the main driver',
        'Mid-week (Tue–Thu) flights are 8–12% cheaper on average',
        'DAC→CGP corridor has the highest intra-airline price variance',
    ],
    'Business Recommendation': [
        'Launch advance-booking Eid campaigns at least 6 weeks before departure',
        'OTA search pages: surface airline-tier price comparisons prominently',
        'Revenue management: test incremental non-stop pricing on peak routes',
        'Price-alert features: default to mid-week alternatives for budget travellers',
        'Deploy real-time competitor monitoring for DAC→CGP pricing windows',
    ],
    'Stakeholder': [
        'Marketing / Product',
        'Product / OTA Partners',
        'Revenue Management',
        'Product / Customer Experience',
        'Revenue Management',
    ],
    'Priority': ['High', 'High', 'Medium', 'Medium', 'Low'],
}

pd.DataFrame(insights)